ROOTファイルの入出力には`uproot`パッケージを用います。  
`Colaboratory`標準ではインストールされていないため、各自以下を実行してインストールします。

In [ ]:
!pip install uproot

必要なパッケージを`import`します。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot

`uproot`パッケージを用いて`ROOT`ファイルを開きます。  
`2020_Co_coin.root`ファイルは、フォルダーアイコンをクリックして、スペースにドラッグすることでアップロードしておいてください。

In [ ]:
with uproot.open("2020_Co_coin.root") as f:
    print(f.keys())

In [ ]:
h1_adc0 = f['h1_adc0']
h1_adc1 = f['h1_adc1']
h2_adc = f['h2_adc']
h1_sca = f['h1_sca']

例えば`h1_adc0`の1次元ヒストグラムを`matplotlib`を用いて描画するコードは以下の通りです。

In [ ]:
values, edges = h1_adc1.to_numpy() # binのエントリ数と境界の値を取得
# 描画
plt.stairs(values, edges, label="TH1D") # 描画
plt.xlabel("x")
plt.ylabel("Entries")
plt.xlim(0,1500)


2次元ヒストグラムの場合は次のようにして表示します。  
`vmax`を変えることでエントリーの上限値を設定します。  
例えば`vmax=500`と変更して見てください。

In [ ]:
values, xedges, yedges = h2_adc.to_numpy()
plt.pcolormesh(xedges, yedges, values.T, vmax=1000)
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar(label="Entries")

h1_scaの可視化

In [ ]:
values, edges = h1_sca.to_numpy() # binのエントリ数と境界の値を取得
# 描画
plt.stairs(values, edges, label="TH1D") # 描画
plt.xlabel("x")
plt.ylabel("Entries")

`TTree`を`pandas Dataframe`として取得します。

In [ ]:
t = f["raw_data"] # tree 取得
df = t.arrays(library="pd") # pandas dataframeに変換

In [ ]:
df.head()

選別条件を課してヒストグラムを作成

In [ ]:
df_sel = df.query('(500 < adc1) & (adc1<800) & (adc0>300)')

In [ ]:
_ = plt.hist(df_sel.adc0,bins=2000,range=(0,2000))

フィッティングには`curve_fit`関数を用います。

In [ ]:
from scipy.optimize import curve_fit

モデル関数を定義します。

In [ ]:
from scipy.special import erfc

def func(x,
         p0, p1, p2, p3, p4, p5, p6, p7, p8, p9, p10):
    g0 = p0 * np.exp(-(x - p1)**2 / (2 * p2**2))
    g1 = p3 * np.exp(-(x - p4)**2 / (2 * p5**2))

    a1 = 0.5 * (p6 * (x**2 + p9**2) + p7 * x + p8)
    b1 = -(p9 / np.sqrt(2 * np.pi)) * p6 * (x + p10) + p7
    c1 = a1 * erfc((x - p10) / (np.sqrt(2) * p9)) \
       + b1 * np.exp(-(x - p10)**2 / (2 * p9**2))

    return g0 + g1 + c1

以下でフィッティングを行います。

In [ ]:
# ヒストグラムからエントリと中心値を取得
counts, edges = np.histogram(df_sel.adc0, bins=2000, range=(0, 2000))
centers = 0.5 * (edges[:-1] + edges[1:])

# 0 count bin を除外
mask = counts > 0
xfit = centers[mask]
yfit = counts[mask]

# 初期値
p0_init = [ 300, 1100, 10, 100, 1200, 20,
           0.0, -0.3, 120.0, 100.0, 900.0]
# フィットを実行
popt, pcov = curve_fit( func, xfit, yfit, p0=p0_init)
perr = np.sqrt(np.diag(pcov))
print("Best-fit parameters:")
for i, (v, e) in enumerate(zip(popt, perr)):
    print(f"p[{i}] = {v:.6g} ± {e:.3g}")

# 描画
xdraw = np.linspace(0, 2000, 5000)

plt.figure(figsize=(8, 5))
plt.hist(df_sel.adc0, bins=2000, range=(0, 2000), histtype="step", label="data")
plt.plot(xdraw, func(xdraw, *popt), label="curve_fit")
plt.xlabel("adc0")
plt.ylabel("Entries / bin")


# コンプトン端のエネルギーの計算

In [ ]:
energies = [1.173,1.333] # MeV

In [ ]:
for hnu in energies:
    Emax = hnu*(2*hnu/0.511)/(1+2*hnu/0.511)
    print(Emax)

$h\nu = 1.173$ MeVのコンプトン端を850chとして$E_\gamma$とチャンネルの対応関係を図示する。

In [ ]:
energies = np.array([0.963,1.173,1.333],dtype="float")
chs = np.array([850,1056,1181],dtype="float")
p, _ = np.polyfit(energies, chs, 1, cov=True)
a, b = p
xdraw = np.linspace(energies.min(), energies.max(), 200)
ydraw = a * xdraw + b
plt.scatter(energies, chs, label="data")
plt.plot(xdraw, ydraw, label=f"fit: ch = {a:.2f} E + {b:.2f}", color='r')
plt.xlabel("Energy")
plt.ylabel("Channel")